# Notebook 01: Korelacijske mreže i mrežne mjere

**Projekt:** Financijske mreže na ZSE (CROBEX, 2004–2026)
**Teme:** Korelacijske matrice, P-prag filtriranje, mjere mrežne topologije
**Kolegiji:** Financijsko modeliranje, Analiza podataka, Uvod u mrežne znanosti

## Kontekst

Financijske mreže konstruiramo iz **matrice korelacija** dionica.
Svaka dionica je čvor; brid postoji samo ako je korelacija statistički značajna
(P-prag pristup, Xu et al. 2017).

Analiziramo **10 mrežnih mjera** koje opisuju strukturu tržišta u svakom vremenskom prozoru:

| Mjera | Opis |
|-------|------|
| M1 LCC | Frakcija dionica u najvećoj komponenti NG mreže |
| M2 APL | Prosječna duljina najkraćeg puta |
| M3 MeanDeg | Prosječni stupanj čvora |
| M4 NCom | Broj zajednica (samo u stresnim periodima) |
| M5 Mod | Modularnost (snaga zajednica) |
| M6 AbsRat | Apsorcijski omjer (sistemski rizik) |
| M7 Close | Prosječna centralnost blizine |
| M8 EigMn | Prosječna eigenvector centralnost |
| M9 EigMx | Maksimalna eigenvector centralnost (PG mreža) |
| M10 PCorr | Prosječna parcijalna korelacija |


In [ ]:
# Postavljanje okolisa (automatski detektira Google Colab)
import sys
if "google.colab" in sys.modules:
    import subprocess, os
    subprocess.run(["git", "clone", "--branch", "teaching",
                    "https://github.com/svlah-sketch/FinNet-teaching.git", "/content/FinNet-teaching"], check=True)
    os.chdir("/content/FinNet-teaching-teaching/notebooks")
    print("Colab: repozitorij kloniran.")
else:
    print("Lokalno pokretanje — nema potrebe za klonom.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

DATA_DIR = Path("../sample_data")

# Učitaj mrežne mjere (W90 — prozori od 90 dana)
metrics = pd.read_csv(DATA_DIR / "sample_metrics_W90.csv",
                      index_col="window_end", parse_dates=True)

print(f"Oblik podataka: {metrics.shape}")
print(f"Vremenski raspon: {metrics.index.min().date()} do {metrics.index.max().date()}")
print()
print(metrics.head())

## 1. Deskriptivna statistika mrežnih mjera

In [ ]:
desc = metrics.describe().T
desc["% missing"] = metrics.isna().mean() * 100
print(desc[["mean","std","min","max","% missing"]].round(3).to_string())

## 2. Vremenska serija odabranih mjera

Vizualiziramo M1 (LCC frakcija), M3 (prosječni stupanj) i M6 (apsorcijski omjer).
Označeni su periodi tri financijske krize.


In [ ]:
CRISES = {
    "GFC":     ("2007-10-01", "2009-09-30"),
    "EU_DEBT": ("2011-04-01", "2012-09-30"),
    "COVID":   ("2020-02-20", "2021-03-31"),
}

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
cols = ["M1_LCC", "M3_MeanDeg", "M6_AbsRat"]
titles = ["M1: LCC frakcija", "M3: Prosječni stupanj", "M6: Apsorcijski omjer"]
colors = ["steelblue", "darkorange", "forestgreen"]

for ax, col, title, color in zip(axes, cols, titles, colors):
    ax.plot(metrics.index, metrics[col], color=color, lw=1.5, label=col)
    for name, (start, end) in CRISES.items():
        ax.axvspan(start, end, alpha=0.12, color="red", label=name if col == cols[0] else "")
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

axes[0].legend(loc="upper right")
fig.suptitle("Mrežne mjere CROBEX-a kroz vrijeme (W90)", fontsize=13)
plt.tight_layout()
plt.savefig("notebook01_figure.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Korelacijska matrica mjera

Koliko su mjere međusobno korelirane? Visoka korelacija sugerira da mjere
hvate iste aspekte tržišne strukture.


In [ ]:
corr = metrics.corr(method="spearman")

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Spearman korelacija između mrežnih mjera (W90)")
plt.tight_layout()
plt.savefig("notebook01_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Razdioba mjera: krize vs. mirna razdoblja

Uspoređujemo raspodjele M1 (LCC) i M3 (stupanj) za krizna i mirna razdoblja.
Vizualni pregled — formalni statistički testovi su u Notebooku 02.


In [ ]:
revs_dates = pd.read_csv(DATA_DIR / "Revisions.csv", sep=";", encoding="cp1250")
revs_dates["start_date"] = pd.to_datetime(revs_dates["start_date"], dayfirst=True)
revs_dates["end_date"]   = pd.to_datetime(revs_dates["end_date"],   dayfirst=True)

def is_crisis(date):
    for start, end in CRISES.values():
        if pd.Timestamp(start) <= date <= pd.Timestamp(end):
            return True
    return False

metrics["kriza"] = metrics.index.map(is_crisis)
print(f"Kriznih prozora: {metrics['kriza'].sum()} / {len(metrics)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["M1_LCC", "M3_MeanDeg"]):
    for label, group in metrics.groupby("kriza")[col]:
        ax.hist(group.dropna(), bins=15, alpha=0.6,
                label="Kriza" if label else "Mirno", density=True)
    ax.set_title(col)
    ax.set_xlabel("Vrijednost mjere")
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle("Razdioba mjera: krize vs. mirna razdoblja")
plt.tight_layout()
plt.show()
metrics.drop(columns=["kriza"], inplace=True)

---

## Zadaci za studente

**1.** Koji par mjera ima najveću apsolutnu Spearman korelaciju? Zašto biste to
   očekivali s obzirom na njihove definicije?

**2.** Dodajte u vremenski graf M9 (EigMx — PG mreža). Ima li suprotan trend od M1?
   Što to govori o ulozi "hub" dionica u mirnim i kriznim periodima?

**3.** Koliko prozora pripada svakoj krizi (GFC, EU_DEBT, COVID)?
   Koristite `metrics.index` i datume kriza iz rječnika `CRISES`.

**4.** (Napredni) M4 i M8 imaju mnogo NaN vrijednosti. Pogledajte `% missing` iz
   deskriptivne statistike. Zašto bi te mjere bile dostupne samo u *nekim* prozorima?
   (Hint: NG-0.001 prag)
